# **PANDAS VS. POLARS**

In [1]:
!pip install polars

In [2]:
import time
from pathlib import Path
import pandas as pd
import polars as pl
import statistics

In [3]:
# URLs dos datasets
CATS_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/MASS/cats.csv"

# Diretório padrão do Colab
DATA_DIR = Path("/content")

def download_if_needed(url: str, out_path: Path):
    if out_path.exists():
        return
    df = pd.read_csv(url)
    df.to_csv(out_path, index=False)
    print(f"Arquivo salvo em: {out_path}")

def make_big_csv(input_csv: Path, out_csv: Path, repeat: int = 5000):
    if out_csv.exists():
        return
    df = pd.read_csv(input_csv)
    big = pd.concat([df] * repeat, ignore_index=True)
    big.to_csv(out_csv, index=False)
    print(f"CSV grande criado: {out_csv}")
    print(f"Linhas: {len(big):,}")

def bench_loads(csv_path: Path, repeats: int = 15):
    # Warmup (evita medir custo inicial)
    pd.read_csv(csv_path)
    pl.read_csv(csv_path)

    pd_times = []
    pl_times = []

    for _ in range(repeats):
        t0 = time.perf_counter()
        pd.read_csv(csv_path)
        pd_times.append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        pl.read_csv(csv_path)
        pl_times.append(time.perf_counter() - t0)

    print("\n==============================")
    print(f"Arquivo: {csv_path.name}")
    print(f"Tamanho: {csv_path.stat().st_size/1e6:.2f} MB")
    print(f"Repetições: {repeats}")
    print(
        f"Pandas  -> mediana: {statistics.median(pd_times):.4f}s | média: {sum(pd_times)/len(pd_times):.4f}s"
    )
    print(
        f"Polars  -> mediana: {statistics.median(pl_times):.4f}s | média: {sum(pl_times)/len(pl_times):.4f}s"
    )
    print("==============================")

# ---------- GATOS ----------
cats_csv = DATA_DIR / "cats.csv"
cats_big = DATA_DIR / "cats_big.csv"

download_if_needed(CATS_URL, cats_csv)
make_big_csv(cats_csv, cats_big, repeat=5000)

bench_loads(cats_csv)
bench_loads(cats_big)



Arquivo salvo em: /content/cats.csv
CSV grande criado: /content/cats_big.csv
Linhas: 720,000

Arquivo: cats.csv
Tamanho: 0.00 MB
Repetições: 15
Pandas  -> mediana: 0.0033s | média: 0.0039s
Polars  -> mediana: 0.0007s | média: 0.0017s

Arquivo: cats_big.csv
Tamanho: 9.95 MB
Repetições: 15
Pandas  -> mediana: 0.4944s | média: 0.5570s
Polars  -> mediana: 0.2126s | média: 0.2567s


# **Pydantic**

In [4]:
!pip install pydantic

In [5]:
import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field, ValidationError

class Cat(BaseModel):
    Sex: Literal["F", "M"]
    Bwt: float = Field(gt=0, description="Peso do gato > 0")
    Hwt: float = Field(gt=0, description="O peso do coração deve ser > 0")


In [6]:
#Pegamos uma linha real do dataset
df = pd.read_csv("/content/cats.csv")
row = df.iloc[0].to_dict()

print("✅ Linha real (antes):", row)

# 2) simula sujeira/erro que poderia vir de uma fonte externa
row["Sex"] = "X"     # inválido
row["Bwt"] = -2.5    # inválido

print("⚠️ Linha alterada (simulação):", row)




✅ Linha real (antes): {'rownames': 1, 'Sex': 'F', 'Bwt': 2.0, 'Hwt': 7.0}
⚠️ Linha alterada (simulação): {'rownames': 1, 'Sex': 'X', 'Bwt': -2.5, 'Hwt': 7.0}


In [7]:
# 3) valida com Pydantic
try:
    Cat(**row)
except ValidationError as e:
    print("🚫 Dado inválido! Motivos:")
    for err in e.errors():
        campo = err["loc"][0]
        msg = err["msg"]
        valor = err.get("input", None)
        print(f"- {campo}: {msg} | valor recebido: {valor}")

🚫 Dado inválido! Motivos:
- Sex: Input should be 'F' or 'M' | valor recebido: X
- Bwt: Input should be greater than 0 | valor recebido: -2.5


# **Rich**

In [8]:
!pip install rich

In [9]:
from rich.console import Console
from rich.table import Table
from rich.progress import track
import time

console = Console()

# 1) Tabela
table = Table(title="Teste do Café com Dados & Gatos ☕🐱")
table.add_column("Etapa", style="bold")
table.add_column("Status")

table.add_row("Baixar dados", "OK")
table.add_row("Limpar dados", "OK")
table.add_row("Treinar modelo", "Rodando...")

console.print(table)

# 2) Barra de progresso
for _ in track(range(30), description="Processando..."):
    time.sleep(0.05)

console.print("[bold green]Finalizado![/bold green]")


Teste do Café com Dados & Gatos
             ☕🐱              
┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Etapa          ┃ Status     ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ Baixar dados   │ OK         │
│ Limpar dados   │ OK         │
│ Treinar modelo │ Rodando... │
└────────────────┴────────────┘

Output()

Finalizado!

# **OPTUNA**

In [4]:
!pip install optuna scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 12.8 MB/s eta 0:00:00


In [9]:
import optuna
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score


In [10]:
# 1) Dataset Breast Cancer (scikit-learn)
X, y = load_breast_cancer(return_X_y=True)

# 2) Cross-validation estratificada (melhor para classificação)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    # Optuna sugere hiperparâmetros
    n_estimators = trial.suggest_int("n_estimators", 100, 600)
    max_depth = trial.suggest_int("max_depth", 2, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 15)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )

    score = cross_val_score(model, X, y, cv=cv, scoring="accuracy").mean()
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)  # pode subir pra 80/100 depois

print("Melhor acurácia:", study.best_value)
print("Melhores parâmetros:", study.best_params)


[I 2026-01-16 21:26:07,635] A new study created in memory with name: no-name-33eab79e-47b2-47d6-bfd5-34a635470f6d
[I 2026-01-16 21:26:14,224] Trial 0 finished with value: 0.9508150908244062 and parameters: {'n_estimators': 429, 'max_depth': 25, 'min_samples_split': 7, 'min_samples_leaf': 9, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9508150908244062.
[I 2026-01-16 21:26:22,657] Trial 1 finished with value: 0.9525694767893185 and parameters: {'n_estimators': 408, 'max_depth': 21, 'min_samples_split': 11, 'min_samples_leaf': 4, 'max_features': 'log2'}. Best is trial 1 with value: 0.9525694767893185.
[I 2026-01-16 21:26:24,938] Trial 2 finished with value: 0.9508150908244062 and parameters: {'n_estimators': 101, 'max_depth': 17, 'min_samples_split': 12, 'min_samples_leaf': 8, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.9525694767893185.
[I 2026-01-16 21:26:35,575] Trial 3 finished with value: 0.9490607048594939 and parameters: {'n_estimators': 504, 'max_depth': 5, '

Melhor acurácia: 0.9560782487191428
Melhores parâmetros: {'n_estimators': 308, 'max_depth': 24, 'min_samples_split': 2, 'min_samples_leaf': 6, 'max_features': 'sqrt'}


In [11]:


vis.plot_optimization_history(study)


In [12]:
from optuna.visualization import plot_param_importances
plot_param_importances(study)
